# PROJET NEOVOLT - DETECTION DE FRAUDE
## CPDIA Data Scientist

Ce notebook reprend et explique un pipeline complet de détection de fraude sur des données de consommation électrique.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression


## Chargement des données

In [2]:
DATA_PATH = Path("donnees")

consommation = pd.read_csv(DATA_PATH / "releves_consommation.csv")
compteurs = pd.read_csv(DATA_PATH / "compteurs.csv")
clients = pd.read_csv(DATA_PATH / "clients.csv")
fraudes = pd.read_csv(DATA_PATH / "cas_fraude_confirmes.csv")

print("Fichiers chargés")

Fichiers chargés


## Conversion des dates

In [3]:
consommation["date"] = pd.to_datetime(consommation["date"], errors="coerce")
clients["date_entree"] = pd.to_datetime(clients["date_entree"], errors="coerce")
compteurs["date_pose"] = pd.to_datetime(compteurs["date_pose"], errors="coerce")
fraudes["date_detection"] = pd.to_datetime(fraudes["date_detection"], errors="coerce")

## Nettoyage des données

In [ ]:
consommation = consommation.drop_duplicates()

consommation["consommation_kwh"] = pd.to_numeric(consommation["consommation_kwh"], errors="coerce")

consommation = consommation[
    consommation["consommation_kwh"].isna()
    | (consommation["consommation_kwh"] >= 0)
]

consommation["consommation_kwh"] = consommation["consommation_kwh"].interpolate()
consommation = consommation.dropna(subset=["consommation_kwh"])

clients["nb_personnes_foyer"] = clients["nb_personnes_foyer"].fillna(0)
clients["surface_m2"] = clients["surface_m2"].fillna(clients["surface_m2"].median())

compteurs = compteurs[compteurs["statut"] == "actif"]

## Fusion des données

In [5]:
if "zone" in compteurs.columns:
    compteurs = compteurs.drop(columns=["zone"])

df = consommation.merge(compteurs, on="id_pdl", how="left")
df = df.merge(clients, on="id_client", how="left")

print(df.shape)

(510668, 17)


## Feature Engineering

In [6]:
df = df.sort_values(["id_pdl", "date"])

df["conso_j_1"] = df.groupby("id_pdl")["consommation_kwh"].shift(1)

df["moyenne_7j"] = df.groupby("id_pdl")["consommation_kwh"].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)

df["std_7j"] = df.groupby("id_pdl")["consommation_kwh"].transform(
    lambda x: x.rolling(7, min_periods=1).std()
)

df["variation_pct"] = df.groupby("id_pdl")["consommation_kwh"].pct_change()

df["conso_par_m2"] = df["consommation_kwh"] / df["surface_m2"].replace(0, np.nan)

df["jour_semaine"] = df["date"].dt.dayofweek
df["mois"] = df["date"].dt.month
df["weekend"] = (df["jour_semaine"] >= 5).astype(int)

df.replace([np.inf, -np.inf], np.nan, inplace=True)


## Création du label fraude

In [7]:
fraudes_ids = set(fraudes["id_pdl"].unique())

df["fraude"] = df["id_pdl"].isin(fraudes_ids).astype(int)

print(df["fraude"].value_counts())

fraude
0    493151
1     17517
Name: count, dtype: int64


## Baseline anomaly detection

In [8]:
df["baseline_anomalie"] = np.where(
    df["consommation_kwh"] >
    (df["moyenne_7j"] + 3 * df["std_7j"].fillna(0)),
    1,
    0
)

print(df["baseline_anomalie"].sum())

0


## Dataset ML

In [9]:
features = [
    "consommation_kwh",
    "conso_j_1",
    "moyenne_7j",
    "std_7j",
    "variation_pct",
    "conso_par_m2",
    "jour_semaine",
    "mois",
    "weekend",
    "surface_m2",
    "nb_personnes_foyer",
    "puissance_souscrite_kva"
]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
y = df["fraude"]

## Equilibrage de Dataset

In [10]:
df_majority = df[df["fraude"] == 0]
df_minority = df[df["fraude"] == 1]

n_minority = len(df_minority)

df_majority_downsampled = df_majority.sample(n=n_minority, random_state=42)

df_balanced = pd.concat([df_majority_downsampled, df_minority])

df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced["fraude"].value_counts())

fraude
0    17517
1    17517
Name: count, dtype: int64


In [11]:
X = df_balanced[features].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
y = df_balanced["fraude"]

## Train/Test split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(28027, 12) (7007, 12)


In [13]:

print("Distribution TRAIN")
train_dist = pd.Series(y_train).value_counts().sort_index()
print(train_dist)
print("Total train:", len(y_train))

print("\n Distribution TEST")
test_dist = pd.Series(y_test).value_counts().sort_index()
print(test_dist)
print("Total test:", len(y_test))

Distribution TRAIN
fraude
0    14013
1    14014
Name: count, dtype: int64
Total train: 28027

 Distribution TEST
fraude
0    3504
1    3503
Name: count, dtype: int64
Total test: 7007


## Modèles

In [14]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=300, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier()
}

results = []

## Entraînement et évaluation

In [15]:
for name, model in models.items():
    print(name)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = y_pred
    report = classification_report(y_test, y_pred, output_dict=True)
    auc = roc_auc_score(y_test, y_proba)

    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print("AUC:", auc)

    results.append({
        "modele": name,
        "precision": report["1"]["precision"],
        "recall": report["1"]["recall"],
        "f1": report["1"]["f1-score"],
        "auc": auc
    })

Logistic Regression


c:\Users\airac\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[[2125 1379]
 [1833 1670]]
              precision    recall  f1-score   support

           0       0.54      0.61      0.57      3504
           1       0.55      0.48      0.51      3503

    accuracy                           0.54      7007
   macro avg       0.54      0.54      0.54      7007
weighted avg       0.54      0.54      0.54      7007

AUC: 0.5574634657573352
Random Forest
[[3132  372]
 [ 269 3234]]
              precision    recall  f1-score   support

           0       0.92      0.89      0.91      3504
           1       0.90      0.92      0.91      3503

    accuracy                           0.91      7007
   macro avg       0.91      0.91      0.91      7007
weighted avg       0.91      0.91      0.91      7007

AUC: 0.9747324781628792
Gradient Boosting
[[3208  296]
 [  37 3466]]
              precision    recall  f1-score   support

           0       0.99      0.92      0.95      3504
           1       0.92      0.99      0.95      3503

    accuracy         

## Comparaison finale

In [16]:
results_df = pd.DataFrame(results)
results_df.sort_values("f1", ascending=False)

,modele,precision,recall,f1,auc
2,Gradient Boosting,0.921318,0.989438,0.954164,0.989057
1,Random Forest,0.896839,0.923209,0.909833,0.974732
0,Logistic Regression,0.547721,0.476734,0.509768,0.557463


## Sauvegarde du modele 

In [17]:
import joblib

# supposons que ton meilleur modèle est déjà entraîné :
best_model = models["Gradient Boosting"]

# sauvegarde
joblib.dump(best_model, "modele_fraude.pkl")

print("Modèle sauvegardé ")

Modèle sauvegardé 


## Test

In [19]:
# charger le modèle
model = joblib.load("modele_fraude.pkl")

print("Modèle chargé ✔")

# exemple de donnée
new_data = pd.DataFrame([{
    "consommation_kwh": 120,
    "conso_j_1": 110,
    "moyenne_7j": 100,
    "std_7j": 15,
    "variation_pct": 0.18,
    "conso_par_m2": 2.5,
    "jour_semaine": 2,
    "mois": 6,
    "weekend": 0,
    "surface_m2": 50,
    "nb_personnes_foyer": 3,
    "puissance_souscrite_kva": 6
}])

# prédiction
pred = model.predict(new_data)[0]

print("Prédiction :", pred)

if hasattr(model, "predict_proba"):
    proba = model.predict_proba(new_data)[0][1]
    print("Probabilité fraude :", proba)

Modèle chargé ✔
Prédiction : 0
Probabilité fraude : 0.1056471320450176
